In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:500])

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {ch:i for i,ch in enumerate(chars)}
idx_to_char = {i:ch for i,ch in enumerate(chars)}

print("Vocabulary size:", vocab_size)

In [ ]:
seq_length = 40
step = 3

input_sequences = []
target_chars = []

for i in range(0, len(text) - seq_length, step):
    input_sequences.append(text[i:i+seq_length])
    target_chars.append(text[i+seq_length])

print("Sequences:", len(input_sequences))

In [ ]:
X = np.zeros((len(input_sequences), seq_length, vocab_size), dtype=np.float32)
y = np.zeros((len(input_sequences), vocab_size), dtype=np.float32)

for i, seq in enumerate(input_sequences):
    for t, char in enumerate(seq):
        X[i, t, char_to_idx[char]] = 1
    y[i, char_to_idx[target_chars[i]]] = 1

In [ ]:
X = torch.tensor(X)
y = torch.tensor(y)

In [ ]:
class SimpleRNN(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):

        out, hidden = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)

        return out

In [ ]:
hidden_size = 128

model = SimpleRNN(vocab_size, hidden_size, vocab_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

In [ ]:
v=epochs = 10
batch_size = 128

for epoch in range(epochs):

    for i in range(0, len(X), batch_size):

        inputs = X[i:i+batch_size]
        targets = torch.argmax(y[i:i+batch_size], dim=1)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()


In [ ]:
import random

def generate_text(model, seed_text, word_count=100):

    model.eval()

    generated = seed_text

    current_text = seed_text[-seq_length:]

    words = generated.split()

    while len(words) < word_count:

        x_pred = np.zeros((1, seq_length, vocab_size))

        for t, char in enumerate(current_text):
            if char in char_to_idx:
                x_pred[0, t, char_to_idx[char]] = 1

        x_pred = torch.tensor(x_pred, dtype=torch.float32)

        with torch.no_grad():
            preds = model(x_pred)

        probs = torch.softmax(preds, dim=1).numpy().ravel()

        next_index = np.random.choice(len(chars), p=probs)
        next_char = idx_to_char[next_index]

        generated += next_char

        current_text = generated[-seq_length:]

        words = generated.split()

    return generated

In [ ]:
seed = "ROMEO:"

generated_text = generate_text(model, seed, 100)

print(generated_text)